# Etapa 1: Selección y caracterización del dataset NYC TLC Yellow Taxi

En esta etapa se caracteriza el dataset de viajes en taxi amarillo de la ciudad de Nueva York durante los años 2024 y 2025, publicado por la New York City Taxi and Limousine Commission (TLC). El dataset contiene registros transaccionales de viajes con atributos de fechas, distancias, tarifas y zonas de origen y destino, lo cual lo hace adecuado para el análisis con PySpark en un entorno de Big Data.

Se utilizan dos años completos (2024 y 2025) para garantizar un volumen total superior a 1 GB y para habilitar análisis comparativo interanual. El año 2026, parcialmente publicado al momento de este trabajo, se reserva como conjunto de validación temporal para las etapas posteriores del proyecto.

El notebook está pensado para ejecutarse de forma portable, tanto localmente como en Google Colab. Todas las rutas de datos son relativas al notebook (`./data/raw`), por lo que cualquier integrante del equipo puede clonar el repositorio y ejecutar las celdas sin ajustes adicionales.

## 0. Configuración del entorno

Este notebook usa las siguientes librerías de Python:

- **pyspark**: motor de cómputo distribuido.
- **findspark**: bootstrap de la sesión local de PySpark.
- **pandas**: se utiliza únicamente para mostrar tablas pequeñas ya agregadas (resumen estadístico, conteo de nulos, matriz de correlación). En este notebook nunca se invoca `.toPandas()` sobre el DataFrame completo de viajes.

La siguiente celda instala las tres librerías. Es idempotente: si ya están instaladas (por ejemplo en `env-pyspark`), pip las reportará como *already satisfied*. En Google Colab se instalan desde cero.

Adicionalmente, si se ejecuta en Google Colab, hay que instalar Java (necesario para correr la JVM de Spark). La celda correspondiente está comentada al final de esta sección y solo se ejecuta en Colab.

In [ ]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas matplotlib

In [ ]:
# Solo en Google Colab: descomentar para instalar Java (la JVM que ejecuta Spark).
# Localmente con env-pyspark esta línea no es necesaria.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null

## 1. Descarga de datos

Los archivos se obtienen directamente del CDN oficial del TLC en formato Parquet. Desde 2022 el TLC distribuye los registros de viajes en Parquet de manera nativa, por su mejor compresión y lectura columnar respecto a CSV. La descarga se realiza de forma idempotente: si el archivo ya existe en disco, se omite, lo cual permite re-ejecutar el notebook sin volver a bajar los datos.

In [ ]:
from pathlib import Path
import subprocess

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
DATA_DIR = Path("data/raw")
YEARS = [2024, 2025]
LOOKUP_FILE = "taxi_zone_lookup.csv"

In [ ]:
def download_if_missing(download_url, target_path):
    """Descarga `download_url` a `target_path` solo si `target_path` no existe.

    Devuelve un string con el estado: 'skip', 'ok' o 'error: <mensaje>'.
    """
    if target_path.exists():
        return "skip"

    target_path.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["curl", "-sSL", "-o", str(target_path), download_url],
        capture_output=True,
        timeout=900,
    )

    if result.returncode != 0:
        return f"error: curl exit {result.returncode}"

    return "ok"

In [ ]:
# Parquets mensuales de viajes
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{CDN_BASE}/trip-data/{filename}"
        target = DATA_DIR / filename
        status = download_if_missing(url, target)
        print(f"{status:>6}  {filename}")

# Tabla de referencia de zonas de taxi
url = f"{CDN_BASE}/misc/{LOOKUP_FILE}"
target = DATA_DIR / LOOKUP_FILE
status = download_if_missing(url, target)
print(f"{status:>6}  {LOOKUP_FILE}")

## 2. Resumen de archivos descargados

Se reporta el inventario y el tamaño en disco. La rúbrica del curso pide un dataset por encima de 1 GB. El formato Parquet ya está comprimido, por lo que el tamaño en disco es menor al equivalente en CSV pero conserva la totalidad de los registros.

In [ ]:
files = sorted(DATA_DIR.glob("*"))
total_bytes = sum(f.stat().st_size for f in files)

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{size_mb:>8.1f} MB   {f.name}")

print()
print(f"Archivos: {len(files)} (esperados: {len(YEARS) * 12 + 1})")
print(f"Tamaño total: {total_bytes / (1024 ** 3):.2f} GB")

parquets = [f for f in files if f.suffix == ".parquet"]
if parquets:
    avg_mb = sum(f.stat().st_size for f in parquets) / len(parquets) / (1024 ** 2)
print(f"Tamaño promedio por Parquet: {avg_mb:.1f} MB")

## 3. Carga del dataset con PySpark

En esta sección se inicializa una sesión local de PySpark y se cargan los 24 archivos Parquet de viajes en un único DataFrame, pasando la lista explícita de rutas a `spark.read.parquet()`.

PySpark implementa evaluación diferida (lazy evaluation): las transformaciones se registran pero no se ejecutan hasta que se invoca una acción como `count()` o `show()`. Esto permite optimizar el plan de ejecución y distribuir el trabajo entre particiones. La carga real de datos ocurre cuando se ejecuta la primera acción, no cuando se define el DataFrame.

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()

# Sube el umbral del log de planes (default 25). Algunas agregaciones de la sección 5
# generan más de 25 expresiones (14 columnas con min y max = 28), lo que dispara un
# WARN benigno que solo trunca el plan en el log, no la computación.
spark.conf.set("spark.sql.debug.maxToStringFields", 100)

print(f"Spark versión: {spark.version}")

### Lista explícita de archivos y unificación de esquemas

En lugar de pasar un patrón glob (`yellow_tripdata_*.parquet`), construimos la lista explícita de rutas con `Path.glob()` y la entregamos a `spark.read.parquet(*paths)`. Esto evita un warning interno que dispara el patrón glob al verificar la existencia de un directorio de metadatos de structured streaming, y deja el output del notebook limpio para el entregable.

Adicionalmente se habilita la opción `mergeSchema=True`. NYC TLC introdujo la columna `cbd_congestion_fee` a partir del 5 de enero de 2025 (cargo por la zona de descongestión central de Manhattan). Los archivos de 2024 no la incluyen, los de 2025 sí. Sin esta opción, Spark toma el esquema del primer archivo y descarta silenciosamente columnas que aparecen en archivos posteriores. Con `mergeSchema=True`, Spark inspecciona el esquema de todos los archivos y construye la unión: los registros de 2024 quedan con `cbd_congestion_fee = null` y los de 2025 con su valor real. Referencia: https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging

In [ ]:
# Lista explícita de rutas a los 24 archivos mensuales
parquet_paths = sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))

# mergeSchema=True para preservar cbd_congestion_fee (presente solo desde 2025-01-05)
df = spark.read.option("mergeSchema", "true").parquet(*parquet_paths)

print(f"Archivos cargados: {len(parquet_paths)}")

### Introspección con printSchema()

El método `printSchema()` muestra el árbol completo de tipos del DataFrame, incluyendo:
- Nombre de cada columna
- Tipo de datos (StringType, LongType, DoubleType, TimestampType, etc.)
- Indicador de nullabilidad: `true` si la columna puede contener nulos, `false` si no.

Esto es más informativo que acceder a `df.columns` o `df.dtypes` directamente. Referencia: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.printSchema.html

In [ ]:
df.printSchema()

### Acciones y evaluación diferida

`count()` es una acción de Spark que dispara la evaluación completa del plan de ejecución. Durante la primera ejecución, Spark lee todos los archivos Parquet desde disco (aproximadamente 1.4 GB) y cuenta las filas. Este proceso puede tardar varios minutos en una máquina local. El resultado se imprime con formato de separador de miles para mejor legibilidad.

Spark **no** memoriza resultados entre acciones por defecto: si se ejecuta otra acción sobre `df` (otro `count()`, un `summary()`, un `groupBy().agg()`, etc.), Spark vuelve a leer los archivos Parquet desde disco.

Por su parte, `df.rdd.getNumPartitions()` informa cuántas particiones físicas distribuyen el DataFrame. Este número depende del tamaño y cantidad de archivos Parquet leídos, y determina el grado de paralelismo de las operaciones posteriores.

In [ ]:
n_rows = df.count()

print(f"Total de filas: {n_rows:,}")

In [ ]:
print(f"Particiones: {df.rdd.getNumPartitions()}")
print()

zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv"))
)
print("Tabla de referencia de zonas de taxi:")
zones.show(5)

### 3.1 Caracterización del catálogo de zonas

Aunque el archivo `taxi_zone_lookup.csv` pesa solo ~12 KB y no califica como Big Data por sí mismo, es la tabla de dimensiones que da semántica a `PULocationID` y `DOLocationID` en los 90 millones de viajes. Antes de avanzar conviene caracterizarlo: tamaño exacto, esquema (re-leído con `inferSchema=True` para que `LocationID` sea entero y no string), distribución por borough y nivel de servicio, presencia de nulos, y cobertura del rango esperado de IDs.

In [ ]:
from pyspark.sql import functions as F

# 3.1 Caracterización del catálogo de zonas
print(f"Total de zonas: {zones.count()}")
print()

print("Esquema (con inferSchema):")
zones.printSchema()

print("Distribución por Borough:")
zones.groupBy("Borough").count().orderBy(F.desc("count")).show(truncate=False)

print("Distribución por service_zone:")
zones.groupBy("service_zone").count().orderBy(F.desc("count")).show(truncate=False)

print("Conteo de nulos por columna:")
zones.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in zones.columns]).show()

print("Cobertura del rango de IDs (esperado 1 a 265):")
ids_present = {row.LocationID for row in zones.select("LocationID").collect()}
expected = set(range(1, 266))
missing = sorted(expected - ids_present)
extra = sorted(ids_present - expected)
print(f"  IDs ausentes en el catálogo: {missing if missing else 'ninguno'}")
print(f"  IDs fuera del rango 1-265:   {extra if extra else 'ninguno'}")

### Lectura del bloque 3.1

El catálogo cubre las zonas TLC referenciadas por `PULocationID` y `DOLocationID` en `df`. La columna `LocationID` es la clave primaria (debe ser única, sin nulos) y los IDs deben ir de 1 a 265 según el diccionario, donde 264 y 265 corresponden a "Unknown" y "Outside of NYC". La columna `service_zone` agrupa zonas por nivel de servicio (Yellow Zone, Boro Zone, Airports, EWR), lo cual será útil en etapas posteriores cuando se analicen patrones de tarifa o demanda por categoría operativa.

La tabla es lo bastante pequeña como para inspeccionarla por completo (no es un caso de Big Data por sí sola), pero su integridad es crítica: cualquier `PULocationID` o `DOLocationID` en `df` que no exista en este catálogo es un error referencial que se documentará en la etapa de calidad.

### Datos cargados

Quedan disponibles dos DataFrames distribuidos en la sesión de Spark:

- `df`: registros de viajes en taxi amarillo (2024-2025), tamaño en disco ~1.4 GB y ~89.9 millones de filas.
- `zones`: tabla de referencia de zonas de taxi (caracterizada en la sección 3.1).

A continuación se documenta el diccionario de variables (sección 4), se validan los rangos efectivos para justificar refinamiento de tipos (sección 5), y finalmente se aplican los downcasts en sitio sobre `df` (sección 6).

## 4. Diccionario de variables (NYC TLC)

La siguiente tabla documenta cada columna del dataset según el diccionario oficial publicado por la New York City Taxi and Limousine Commission. Esta documentación es la fuente de verdad para los rangos válidos, valores enumerados y semántica de cada campo, y se usará en la sección siguiente para justificar refinamientos de tipos y reglas de validación de calidad.

Referencia: https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf (versión 18 de marzo de 2025)

| Columna | Tipo en Parquet | Descripción | Rango / valores válidos |
|---|---|---|---|
| `VendorID` | integer | Código del proveedor TPEP que generó el registro | enum: 1=Creative Mobile Technologies, 2=Curb Mobility, 6=Myle Technologies, 7=Helix |
| `tpep_pickup_datetime` | timestamp_ntz | Fecha y hora cuando el taxímetro fue activado | dentro del año del archivo |
| `tpep_dropoff_datetime` | timestamp_ntz | Fecha y hora cuando el taxímetro fue desactivado | posterior a `tpep_pickup_datetime` |
| `passenger_count` | long | Número de pasajeros en el vehículo | entero ≥ 0; típicamente 1 a 6 |
| `trip_distance` | double | Distancia del viaje en millas reportada por el taxímetro | ≥ 0 millas |
| `RatecodeID` | long | Código de tarifa aplicado al final del viaje | enum: 1=Standard, 2=JFK, 3=Newark, 4=Nassau/Westchester, 5=Negotiated, 6=Group ride, 99=Null/unknown |
| `store_and_fwd_flag` | string | Indica si el registro se almacenó en memoria del vehículo antes de transmitirse al servidor | enum: 'Y', 'N' |
| `PULocationID` | integer | Zona TLC donde se inició el viaje | entero, generalmente 1 a 263 (más 264 y 265 para zonas desconocidas) |
| `DOLocationID` | integer | Zona TLC donde se finalizó el viaje | igual que `PULocationID` |
| `payment_type` | long | Método de pago | enum: 0=Flex Fare, 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided trip |
| `fare_amount` | double | Tarifa por tiempo y distancia calculada por el taxímetro (USD) | ≥ 0 USD |
| `extra` | double | Extras y recargos diversos (USD) | ≥ 0 USD |
| `mta_tax` | double | Impuesto MTA aplicado según la tarifa metered (USD) | ≥ 0 USD |
| `tip_amount` | double | Propina (solo se registra para pagos con tarjeta de crédito; las propinas en efectivo no aparecen) (USD) | ≥ 0 USD |
| `tolls_amount` | double | Suma total de peajes pagados durante el viaje (USD) | ≥ 0 USD |
| `improvement_surcharge` | double | Recargo de mejora aplicado al inicio del viaje (vigente desde 2015) (USD) | ≥ 0 USD |
| `total_amount` | double | Monto total cobrado al pasajero (no incluye propina en efectivo) (USD) | ≥ 0 USD |
| `congestion_surcharge` | double | Recargo por congestión del estado de NY (USD) | ≥ 0 USD |
| `Airport_fee` | double | Cargo solo para abordajes en LaGuardia y JFK (USD) | ≥ 0 USD |
| `cbd_congestion_fee` | double | Cargo por la zona de descongestión central de Manhattan, vigente desde 2025-01-05 (USD) | ≥ 0 USD; null para registros previos a 2025-01-05 |

## 5. Validación previa al refinamiento de tipos

El esquema actual del dataset está sobre-dimensionado en varias columnas: identificadores enumerados (`VendorID`, `RatecodeID`, `payment_type`) están almacenados como `long` (64 bits) cuando sus valores válidos caben en `byte` (8 bits); las zonas (`PULocationID`, `DOLocationID`) están en `integer` (32 bits) cuando caben en `short` (16 bits); y los montos monetarios están en `double` (64 bits) cuando para tarifas de taxi (típicamente menores a USD 1000) la precisión de `float` (32 bits) es suficiente.

Antes de aplicar cualquier downcast es necesario validar empíricamente que los valores reales en los datos caben en el tipo más estrecho. Esta sección ejecuta tres bloques de validación:

1. **Valores distintos en columnas enumeradas** para confirmar que coinciden con el diccionario oficial.
2. **Mínimos y máximos en columnas numéricas** para confirmar que los rangos efectivos caben en los tipos más estrechos propuestos.
3. **Presencia de la columna `cbd_congestion_fee`** y proporción de nulos por año, para verificar que `mergeSchema=True` la rescató correctamente.

Si las validaciones pasan, se aplicarán los casts en la sección 6 con justificación por columna. Si alguna validación falla, se documenta como hallazgo y se decide si tratarlo como dato sucio en la etapa de calidad o ampliar el tipo destino.

In [ ]:
# 5.1 Valores distintos en columnas enumeradas

enum_cols = ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag"]

for col_name in enum_cols:
    print(f"\n{col_name}:")
    (df.groupBy(col_name)
        .count()
        .orderBy(col_name)
        .show(truncate=False))

### Lectura del bloque 5.1

Cada tabla muestra los valores únicos observados en una columna enumerada y su frecuencia. Se compara contra los valores documentados:

- `VendorID`: deben observarse subconjuntos de {1, 2, 6, 7}.
- `RatecodeID`: subconjuntos de {1, 2, 3, 4, 5, 6, 99}.
- `payment_type`: subconjuntos de {0, 1, 2, 3, 4, 5, 6}.
- `store_and_fwd_flag`: subconjuntos de {'Y', 'N'}.

La presencia de un valor `null` se cuenta como una categoría más y se documenta. Valores fuera del catálogo oficial se registrarán como problema de calidad en el paso siguiente.

In [ ]:
# 5.2 Rangos efectivos de columnas numéricas
numeric_cols = [
    "passenger_count", "trip_distance",
    "PULocationID", "DOLocationID",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "Airport_fee", "cbd_congestion_fee",
]

agg_exprs = []
for c in numeric_cols:
    agg_exprs.append(F.min(c).alias(f"{c}__min"))
    agg_exprs.append(F.max(c).alias(f"{c}__max"))

ranges_row = df.agg(*agg_exprs).first().asDict()

print(f"{'Columna':<25} {'min':>20} {'max':>20}")
for c in numeric_cols:
    mn = ranges_row[f"{c}__min"]
    mx = ranges_row[f"{c}__max"]
    print(f"{c:<25} {str(mn):>20} {str(mx):>20}")

### Lectura del bloque 5.2

La tabla muestra los valores mínimo y máximo observados en cada columna numérica del dataset completo. Se compara cada rango contra el rango del tipo destino propuesto:

| Columna | Tipo destino propuesto | Rango del tipo | Verificación |
|---|---|---|---|
| `passenger_count` | `byte` (8 bits) | -128 a 127 | min ≥ 0 y max ≤ 127 |
| `PULocationID`, `DOLocationID` | `short` (16 bits) | -32 768 a 32 767 | min ≥ 1 y max ≤ 32 767 |
| `trip_distance` | `float` (32 bits) | ~7 dígitos significativos | max menor a 1×10⁷ para preservar precisión decimal |
| Montos en USD (`fare_amount`, `tip_amount`, `total_amount`, etc.) | `float` (32 bits) | ~7 dígitos significativos | max menor a 1×10⁷ |

Si algún max excede el rango del tipo destino, ese tipo se descarta para esa columna o se trata como outlier en la etapa de calidad. Los valores negativos en montos o distancias también se registran: pueden ser anulaciones legítimas (con `RatecodeID` o `payment_type` asociados) o ruido a limpiar.

In [ ]:
# 5.3 Verificación de cbd_congestion_fee tras mergeSchema
print("Columnas en df:")
print(df.columns)
print()

cbd_present = "cbd_congestion_fee" in df.columns
print(f"cbd_congestion_fee presente en el esquema: {cbd_present}")

if cbd_present:
    null_by_year = (
        df.withColumn("year", F.year("tpep_pickup_datetime"))
          .groupBy("year")
          .agg(
              F.count("*").alias("n_filas"),
              F.sum(F.col("cbd_congestion_fee").isNull().cast("int")).alias("n_nulos_cbd"),
          )
          .orderBy("year")
    )
    null_by_year.show(truncate=False)

### Lectura del bloque 5.3

Se confirma que `mergeSchema=True` rescató la columna `cbd_congestion_fee` y se cuantifica la proporción de nulos por año:

- Para `year = 2024`: se espera `n_nulos_cbd = n_filas` (100% de nulos). La columna no existía en los archivos de 2024 y `mergeSchema` la introduce vacía.
- Para `year = 2025`: se esperan algunos nulos durante los primeros días de enero (la tarifa entró en vigor el 5 de enero) y luego mayoría con valor.
- Cualquier `year` distinto a 2024 o 2025 indica fechas fuera del período del dataset, lo cual es un problema de calidad a registrar en el paso siguiente.

## 6. Refinamiento del esquema con downcast de tipos

Las validaciones de la sección 5 confirmaron que los rangos efectivos del dataset caben en tipos más estrechos que los que NYC TLC usa en el Parquet original. Esta sección aplica los downcasts en una sola transformación con `selectExpr` (siguiendo el patrón usado en clases `Ejemplo2.ipynb`), reasignando `df` en sitio.

El motivo del refinamiento no es solamente cosmético. Aunque el archivo Parquet en disco no se modifica, los tipos que Spark expone en su DataFrame se reflejan directamente en la representación interna en memoria: cada fila ocupa menos bytes cuando se materializa. Esto reduce el costo de las operaciones que mueven datos: shuffles entre etapas, broadcast joins y la presión sobre la heap del executor. Para un dataset de ~90 millones de filas, el ahorro agregado es relevante.

Se reasigna `df` en sitio para evitar tener dos referencias al mismo plan lógico ocupando espacio mental: el `df` post-cast es la única fuente de verdad para las etapas siguientes.

### Plan de casts y ahorro estimado

| Columna | Tipo actual (bytes) | Tipo destino (bytes) | Bytes ahorrados / fila | Justificación |
|---|---|---|---|---|
| `VendorID` | int (4) | tinyint (1) | 3 | Enum oficial {1, 2, 6, 7}; max=7 ≤ 127 |
| `passenger_count` | long (8) | tinyint (1) | 7 | Rango observado [0, 9]; max=9 ≤ 127 |
| `RatecodeID` | long (8) | tinyint (1) | 7 | Enum {1-6, 99}; max=99 ≤ 127 |
| `PULocationID` | int (4) | smallint (2) | 2 | Rango [1, 265]; cabe en short [-32k, 32k] |
| `DOLocationID` | int (4) | smallint (2) | 2 | Igual que `PULocationID` |
| `payment_type` | long (8) | tinyint (1) | 7 | Enum {0-6}; max observado=5 ≤ 127 |
| `trip_distance` | double (8) | float (4) | 4 | Distancias en millas; precisión float (~7 dígitos significativos) suficiente para valores reales (<10⁴ mi). Outliers (398k mi) son bogus y se filtrarán en limpieza |
| `fare_amount` | double (8) | float (4) | 4 | Tarifas típicas <USD 1000; float preserva centavos hasta ~USD 10⁵ |
| `extra` | double (8) | float (4) | 4 | Recargos pequeños (<USD 200) |
| `mta_tax` | double (8) | float (4) | 4 | Valores pequeños |
| `tip_amount` | double (8) | float (4) | 4 | Propinas observadas <USD 1000 |
| `tolls_amount` | double (8) | float (4) | 4 | Peajes observados <USD 2000 |
| `improvement_surcharge` | double (8) | float (4) | 4 | Recargo casi fijo |
| `total_amount` | double (8) | float (4) | 4 | Mismo argumento que `fare_amount` |
| `congestion_surcharge` | double (8) | float (4) | 4 | Recargo casi fijo (<USD 5) |
| `Airport_fee` | double (8) | float (4) | 4 | Cargo fijo (<USD 10) |
| `cbd_congestion_fee` | double (8) | float (4) | 4 | Cargo fijo (<USD 5) |

**Total ahorro: 72 bytes / fila** sobre las columnas tipadas.

Sobre 89,892,322 filas, el ahorro agregado es de aproximadamente **6.5 GB en presión de memoria** al materializar el DataFrame (shuffles, broadcast, plan execution). El tamaño en disco del Parquet original no cambia: el ahorro se realiza en tiempo de ejecución, donde Spark deserializa cada fila a su representación interna (Catalyst InternalRow) y los tipos más estrechos ocupan menos memoria por fila.

**Columnas que no se modifican**:

| Columna | Tipo | Razón |
|---|---|---|
| `tpep_pickup_datetime` | timestamp_ntz | Ya es el tipo correcto (sin zona horaria, según convención NYC TLC) |
| `tpep_dropoff_datetime` | timestamp_ntz | Igual |
| `store_and_fwd_flag` | string | Solo 3 valores ('Y', 'N', null); convertir a boolean perdería la categoría null. Se evaluará en la etapa de limpieza |

In [ ]:
# Aplicación de los casts en una sola transformación con selectExpr.
# Patrón inspirado en sem2/classroom/Ejemplo2.ipynb (DDL inline en cada expresión).
# Se reasigna `df` en sitio: la versión refinada reemplaza a la original, no se duplica el dataset.
df = df.selectExpr(
    "cast(VendorID as tinyint) VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "cast(passenger_count as tinyint) passenger_count",
    "cast(trip_distance as float) trip_distance",
    "cast(RatecodeID as tinyint) RatecodeID",
    "store_and_fwd_flag",
    "cast(PULocationID as smallint) PULocationID",
    "cast(DOLocationID as smallint) DOLocationID",
    "cast(payment_type as tinyint) payment_type",
    "cast(fare_amount as float) fare_amount",
    "cast(extra as float) extra",
    "cast(mta_tax as float) mta_tax",
    "cast(tip_amount as float) tip_amount",
    "cast(tolls_amount as float) tolls_amount",
    "cast(improvement_surcharge as float) improvement_surcharge",
    "cast(total_amount as float) total_amount",
    "cast(congestion_surcharge as float) congestion_surcharge",
    "cast(Airport_fee as float) Airport_fee",
    "cast(cbd_congestion_fee as float) cbd_congestion_fee",
)

print("Esquema refinado:")
df.printSchema()

### Esquema refinado

`df` queda reasignado en sitio con los tipos definitivos para el resto del análisis. Las próximas etapas (análisis exploratorio, calidad de datos, muestreo y modelado) operan sobre este esquema.

`df` apunta ahora a un plan lógico que incluye la lectura del Parquet original más las transformaciones de cast. Spark sigue leyendo los archivos en su tipo nativo desde disco y aplica los downcasts en memoria en cada acción.

## 7. Análisis exploratorio descriptivo

Esta sección calcula estadísticas descriptivas distribuidas sobre `df`: resumen numérico, conteo de nulos por columna, top-10 de zonas de origen y destino, patrones temporales (hora, día de semana, mes) y matriz de correlación entre variables numéricas. Cada subsección lleva una explicación del cómputo y una lectura breve del resultado.

### 7.1 Resumen estadístico de variables numéricas

Esta subsección describe la distribución de cada variable continua del dataset. Se separa el cómputo en dos partes para mantener acotado el costo en memoria:

1. `df.summary("count", "mean", "stddev", "min", "max")` calcula los estadísticos básicos en una sola pasada. Se pasan los nombres de los estadísticos de forma explícita para omitir los percentiles del cálculo: con 12 columnas y 90 millones de filas, la versión con percentiles por defecto requiere materializar muchas estructuras intermedias y satura la heap por defecto de la JVM.
2. `approxQuantile(col, probabilities, relativeError)` calcula percentiles columna por columna, usando el algoritmo de Greenwald-Khanna en una sola pasada. Con `relativeError=0.01` el error garantizado es menor al 1% del rango total, suficiente para describir la distribución de tarifas y distancias. Esto se ejecuta en la celda siguiente sobre las cuatro variables más relevantes (`fare_amount`, `trip_distance`, `tip_amount`, `total_amount`) con percentiles 25, 50, 75, 90, 95 y 99.

Para Big Data, `approxQuantile` es preferible a ordenar el DataFrame completo (que requeriría un shuffle global de todos los datos), y es la opción práctica cuando el dataset no cabe en la memoria del driver.

In [ ]:
# Columnas continuas de interés para el resumen estadístico.
# Se excluyen columnas categóricas codificadas como numéricas
# (VendorID, RatecodeID, payment_type, PULocationID, DOLocationID)
# y columnas de timestamp, ya que los estadísticos de media/stddev
# no son interpretables para esos tipos.
continuous_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
]

# summary() con stats explícitos: omitimos percentiles porque su cálculo
# sobre 12 columnas y 90M filas requiere ordenar/aproximar todas a la vez
# y satura la heap por defecto. Los percentiles se calculan en la siguiente
# celda con approxQuantile, columna por columna, que es mucho más barato.
summary_sdf = df.select(continuous_cols).summary("count", "mean", "stddev", "min", "max")

# Transponemos para que cada variable quede en su propia fila y los
# estadísticos como columnas. La tabla resultante tiene 12 filas por 5
# columnas, totalmente apto para .toPandas().
summary_pd = summary_sdf.toPandas().set_index("summary").T
summary_pd.index.name = "variable"

print("Resumen estadístico (transpuesto, una fila por variable):")
print(summary_pd.to_string())

In [ ]:
# Percentiles extendidos para las variables clave de tarifa y distancia.
# approxQuantile devuelve una lista de valores por columna.
# relativeError=0.01 garantiza un error menor al 1% del rango de cada variable.
quantile_cols = ["fare_amount", "trip_distance", "tip_amount", "total_amount"]
probabilities = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
rel_error = 0.01

results = df.approxQuantile(quantile_cols, probabilities, rel_error)

print(f"approxQuantile (relativeError={rel_error})")
print(f"{'Percentil':<12}" + "".join(f"{c:>20}" for c in quantile_cols))
for i, p in enumerate(probabilities):
    row = "".join(f"{results[j][i]:>20.2f}" for j in range(len(quantile_cols)))
    print(f"p{int(p*100):<11}" + row)

#### Interpretación del resumen estadístico

Las distribuciones observadas confirman patrones consistentes con datos transaccionales urbanos y a la vez evidencian la presencia de outliers extremos que se procesarán en la etapa de limpieza:

- **`fare_amount`**: media USD 18.81, mediana USD 14.20 (p50 de approxQuantile). El sesgo positivo es marcado: la media supera la mediana en aproximadamente 32%. La desviación estándar (USD 117.28) es enorme respecto al rango intercuartílico (USD 13.30 entre p25 y p75), señal de que la varianza está dominada por valores extremos. El máximo observado (USD 863,372) y el mínimo (USD -2,261) están fuera de cualquier rango realista para un viaje de taxi.
- **`trip_distance`**: media 5.99 millas, mediana 1.80 millas (sesgo positivo extremo, media equivalente a 3.3 veces la mediana). El p99 es 12.67 millas, pero el máximo es 398,608.62 millas, físicamente imposible (más de 16 vueltas a la circunferencia terrestre). La desviación estándar (555.92) es ~93 veces la media, dominada por unos pocos outliers absurdos.
- **`tip_amount`**: el p25 es 0.00, lo que confirma que al menos un cuarto de los registros tienen propina nula (consistente con pagos en efectivo, que NYC TLC no captura). La mediana es USD 2.38 y la media USD 3.06; el p99 llega a USD 10.93 y el máximo a USD 999.99 (probable tope técnico del sistema).
- **`total_amount`**: hereda los outliers de `fare_amount`. Mediana USD 21.06, máximo USD 863,380 (apenas USD 8 sobre el máximo de `fare_amount`, consistente con la suma de recargos casi fijos sobre la tarifa máxima).
- **`passenger_count`**, **`congestion_surcharge`** y **`Airport_fee`**: el `count` reportado para estas columnas es 74,189,196, menor al total del dataset (89,892,322). La diferencia (15,703,126 filas) corresponde al patrón de nulos del bloque Flex Fare identificado en la sección 5.1.
- **`cbd_congestion_fee`**: solo 48,722,602 valores no nulos (54% del dataset); el resto son los registros previos al 5 de enero de 2025 más los Flex Fare de 2025 sin reporte. La media es USD 0.53 y el max USD 1.75, consistente con un cargo casi fijo.

Los valores mínimos negativos en montos monetarios (`fare_amount`, `total_amount`, `tip_amount`, `tolls_amount`, etc.) corresponden a transacciones anuladas o ajustes de crédito legítimos, no a errores. La etapa de limpieza decidirá si se mantienen, se filtran o se etiquetan como reversiones según el `payment_type` o `RatecodeID` asociados.

### 7.2 Conteo de nulos por columna

Una forma natural de contar nulos sería `df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])`: agregar todas las expresiones en una sola pasada. Sobre 21 columnas y 90 millones de filas, sin embargo, esa estrategia bajo la heap por defecto de la JVM saturaba la memoria intermitentemente. Un experimento posterior con `df.agg(*[F.count(c) for c in df.columns])` produjo el mismo resultado: 20 agregaciones simultáneas exceden la memoria disponible.

Como solución pragmática se separa el conteo en dos pasos:

1. `df.summary("count")`: la primitiva ya validada en la sección 7.1, que cuenta valores no nulos por columna pero omite tipos `timestamp_ntz`.
2. `df.agg(*[F.count(c) for c in ts_cols])`: una agregación adicional restringida a las dos columnas de timestamp que summary deja fuera. Como solo procesa 2 columnas, es muy ligera.

Los nulos por columna se derivan como `n_rows - non_null`, y el porcentaje se calcula contra el total de filas obtenido en la sección 3.

In [ ]:
# El conteo se hace en dos pasos para mantener cada job ligero:
# (1) df.summary('count') sobre todas las columnas que soporta (numéricas
#     y string); este job ya se ejecutó sin problemas en 7.1.
# (2) F.count solo sobre las columnas de tipo timestamp_ntz, que summary
#     omite por defecto. Es una agregación de 2 columnas, muy ligera.
# Combinar 20 agregaciones en un solo job satura la heap por defecto, por
# eso se evita esa vía.
counts_pd = df.summary("count").toPandas().drop(columns=["summary"]).T
counts_pd.columns = ["non_null"]
counts_pd["non_null"] = counts_pd["non_null"].astype(int)

# Columnas de timestamp omitidas por summary: contar por separado.
ts_cols = [c for c, t in df.dtypes if t.startswith("timestamp") and c not in counts_pd.index]
if ts_cols:
    ts_row = df.agg(*[F.count(c).alias(c) for c in ts_cols]).first()
    for c in ts_cols:
        counts_pd.loc[c] = [ts_row[c]]

# Reordenar al orden original del DataFrame.
counts_pd = counts_pd.reindex(df.columns)

counts_pd["nulos"] = n_rows - counts_pd["non_null"]
counts_pd["pct"] = (counts_pd["nulos"] / n_rows * 100).round(2)
counts_pd.index.name = "columna"

print("Nulos por columna:")
print(counts_pd[["nulos", "pct"]].to_string())

#### Interpretación del conteo de nulos

La tabla evidencia tres bloques de nulidad bien definidos:

- **Sin nulos (0% en el dataset completo)**: `VendorID`, `tpep_pickup_datetime`, `tpep_dropoff_datetime`, `trip_distance`, `PULocationID`, `DOLocationID`, `payment_type`, `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge` y `total_amount`. Estas columnas conforman el núcleo operativo del registro de viaje y siempre se reportan.
- **Nulos correlacionados (15,703,126 filas, 17.47% del total)**: `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge` y `Airport_fee` tienen exactamente el mismo número de nulos. Como se identificó en la sección 5.1, este bloque corresponde a viajes con `payment_type = 0` (Flex Fare): para estos viajes el vendedor no reporta los campos del taxímetro tradicionales.
- **Nulos por cobertura temporal (`cbd_congestion_fee`, 41,169,720 filas, 45.80% del total)**: la columna no existía en los archivos de 2024 ni en parte de los Flex Fare de 2025. No son datos faltantes sino ausencia esperada documentada en el diccionario oficial.

La etapa de calidad de datos decidirá la estrategia para los nulos del bloque Flex Fare según las necesidades de cada análisis posterior: imputar, mantener como categoría propia o filtrar.

In [ ]:
# Verificación: ¿son las mismas filas las que tienen las 5 columnas nulas?
# Si el conteo de filas con las 5 simultáneamente nulas se aproxima al 17.47%
# observado en cada columna individual, queda demostrado que es un único
# subconjunto de registros (hipótesis Flex Fare) y no nulos independientes.
flex_cols = ["passenger_count", "RatecodeID", "store_and_fwd_flag",
             "congestion_surcharge", "Airport_fee"]

cond_all = F.col(flex_cols[0]).isNull()
for c in flex_cols[1:]:
    cond_all = cond_all & F.col(c).isNull()

flex_rows = df.filter(cond_all).count()
flex_pct = flex_rows / n_rows * 100

# Conteo individual previamente calculado en counts_pd para una de las 5 cols.
indiv_nulls = int(counts_pd.loc["passenger_count", "nulos"])
indiv_pct = float(counts_pd.loc["passenger_count", "pct"])

print(f"Filas con las 5 columnas Flex Fare nulas simultáneamente: {flex_rows:,} ({flex_pct:.2f}%)")
print(f"Filas con passenger_count nulo (conteo individual):       {indiv_nulls:,} ({indiv_pct:.2f}%)")
print(f"Diferencia: {abs(flex_rows - indiv_nulls):,} filas ({abs(flex_pct - indiv_pct):.2f} pp)")


#### Verificación y caracterización del bloque Flex Fare

**¿Por qué la coincidencia valida la hipótesis?** Si las cinco columnas tuvieran patrones de nulidad independientes, el conteo de filas con las cinco simultáneamente nulas sería mucho menor que el conteo individual (producto de probabilidades). La igualdad exacta entre ambos conteos demuestra que la nulidad está perfectamente correlacionada y proviene de un único evento estructural común a esas filas, no de fallas de captura aisladas.

**¿Qué es Flex Fare?** Es la modalidad tarifaria que la New York City Taxi and Limousine Commission introdujo para alinear los taxis amarillos con plataformas de viaje por aplicación (Curb, Arro y similares). El viaje se solicita y se cotiza desde la app con tarifa acordada por adelantado, y el taxímetro tradicional no se activa durante el trayecto. Se identifica con `payment_type = 0`, fuera del rango 1-6 que documenta el manual TLC clásico. Los campos que produce el medidor (`passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, `Airport_fee`) quedan nulos por diseño operativo, no por fallo de captura. Los campos que sí se reportan al cobro (tarifa, total, distancia, marcas de tiempo, zonas, propina) siguen presentes.

**Decisión de tratamiento por tipo de análisis posterior**:

| Tipo de análisis | Tratamiento de las filas Flex Fare |
|---|---|
| Demanda por hora, día o zona | Incluir (representan demanda real de viajes) |
| Análisis del medidor (`passenger_count`, `RatecodeID`, recargos) | Excluir filtrando por `is_flex_fare = false` |
| Modelado supervisado de tarifa o duración | Incluir con la bandera `is_flex_fare` como variable predictora |
| Análisis de propinas | Ortogonal: el filtro `payment_type = 1` (tarjeta) ya excluye Flex Fare |

La corrección operativa en la Etapa 2 es marcar la bandera `is_flex_fare = (payment_type == 0)` y conservar las filas; cada análisis posterior decide cómo usarlas según la matriz anterior. No corresponde imputar las cinco columnas porque eso inventaría telemetría del medidor que nunca existió.


### 7.3 Top-10 zonas de origen y destino

Los campos `PULocationID` y `DOLocationID` almacenan códigos numéricos del catálogo TLC. Un código como 237 no transmite información geográfica inmediata; en cambio, "Upper East Side North (Manhattan)" sí lo hace. Por eso se realiza un join con el DataFrame `zones` (caracterizado en la sección 3.1) para enriquecer los resultados con las columnas `Borough`, `Zone` y `service_zone`.

El join es eficiente porque `zones` tiene solo 265 filas y Spark lo convierte automáticamente en un broadcast join: la tabla pequeña se copia a cada executor, evitando el shuffle de los 90 millones de filas del DataFrame principal. El `groupBy().count()` se ejecuta antes del join para que el shuffle de agregación opere sobre el DataFrame grande antes de enriquecerlo.

In [ ]:
# Top-10 zonas de origen (PULocationID)
top_pu = (
    df.groupBy("PULocationID")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
    .join(
        zones.select(
            F.col("LocationID").cast("smallint").alias("PULocationID"),
            "Borough",
            "Zone",
            "service_zone",
        ),
        on="PULocationID",
        how="left",
    )
    .orderBy(F.desc("count"))
)

print("Top-10 zonas de origen:")
top_pu.show(truncate=False)

# Top-10 zonas de destino (DOLocationID)
top_do = (
    df.groupBy("DOLocationID")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
    .join(
        zones.select(
            F.col("LocationID").cast("smallint").alias("DOLocationID"),
            "Borough",
            "Zone",
            "service_zone",
        ),
        on="DOLocationID",
        how="left",
    )
    .orderBy(F.desc("count"))
)

print("Top-10 zonas de destino:")
top_do.show(truncate=False)

#### Interpretación de las zonas de mayor demanda

Los rankings observados revelan una asimetría operativa relevante entre orígenes y destinos:

- **Top-10 orígenes**: 8 de las 10 zonas son Yellow Zone de Manhattan (Upper East Side South, Midtown Center, Upper East Side North, Midtown East, Times Sq/Theatre District, Penn Station/Madison Sq West, Lincoln Square East, Murray Hill). Las dos restantes son aeropuertos: **JFK Airport ocupa el primer lugar absoluto con 4,052,450 viajes** y LaGuardia Airport ocupa el lugar 9 con 2,586,847 viajes.
- **Top-10 destinos**: las 10 zonas son exclusivamente Yellow Zone de Manhattan. Ningún aeropuerto aparece en el ranking.

La asimetría se explica por el modelo regulatorio TLC: en NYC, los taxis amarillos tienen acceso preferente a las filas de espera de pasajeros en aeropuertos (origen), mientras que los pasajeros que llegan a aeropuertos suelen llegar en vehículos privados o servicios de viaje compartido. Por eso los aeropuertos dominan los pickups del taxi amarillo y prácticamente desaparecen de los drop-offs.

Las zonas de Manhattan se superponen ampliamente entre ambos rankings: Upper East Side North y South, Midtown Center, Times Sq/Theatre District, Murray Hill, Midtown East y Lincoln Square East aparecen en ambos top-10. Estas zonas funcionan como hubs de circulación, tanto de salida como de llegada. Ningún registro del top-10 corresponde a los códigos 264 (Unknown) o 265 (Outside of NYC), lo que sugiere baja prevalencia de errores de georreferencia entre las zonas de mayor volumen, aunque no descarta su presencia en la cola de la distribución.

### 7.4 Patrones temporales

Para analizar patrones por hora del día, día de semana y mes, primero se deriva la columna `trip_duration_min` a partir de los timestamps de inicio y fin del viaje.

Los timestamps de NYC TLC están almacenados como `timestamp_ntz` (timestamp without timezone): valores de wall-clock sin información de zona horaria. Para calcular la duración entre dos `timestamp_ntz` de manera independiente de la zona horaria del entorno donde se ejecute el notebook, se usa `F.timestamp_diff(unit, start, end)`, función de PySpark 3.5+ que opera directamente sobre los componentes de wall-clock sin convertir a Unix seconds y, por lo tanto, no depende de la zona horaria de la sesión de Spark. Se solicita la diferencia en `SECOND` y se divide entre 60 para obtener minutos en formato decimal. Referencia: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.timestamp_diff.html

Las funciones de extracción temporal relevantes son:
- `F.hour(col)`: extrae la hora del día (0 a 23) desde un timestamp.
- `F.dayofweek(col)`: extrae el día de la semana como entero (1 = domingo, 2 = lunes, ..., 7 = sábado). La convención de Spark sigue el estándar ISO de Java donde 1 es domingo, no lunes.
- `F.month(col)`: extrae el mes (1 a 12).
- `F.year(col)`: extrae el año.

La columna `trip_duration_min` se agrega a `df` con `withColumn`, reasignando la referencia en sitio.

In [ ]:
# Derivar trip_duration_min a partir de los timestamps.
# Se usa F.timestamp_diff, que opera directamente sobre wall-clock y no
# depende de la zona horaria de la sesión de Spark; esto importa porque
# el notebook puede ejecutarse en máquinas con cualquier TZ configurada.
# El primer argumento (unit) es un string Python, no una Column. La función
# devuelve un entero en la unidad solicitada (SECOND); se divide entre 60.0
# para obtener minutos como float.
df = df.withColumn(
    "trip_duration_min",
    F.timestamp_diff(
        "SECOND",
        F.col("tpep_pickup_datetime"),
        F.col("tpep_dropoff_datetime"),
    ) / 60.0,
)

print("Columna trip_duration_min agregada.")
print(f"Total de columnas ahora: {len(df.columns)}")

In [ ]:
# Patrón 1: distribución por hora del día (0-23).
# Muestra conteo, tarifa promedio y duración promedio por hora.
print("Distribución por hora del día:")
(
    df.groupBy(F.hour("tpep_pickup_datetime").alias("hora"))
    .agg(
        F.count("*").alias("viajes"),
        F.round(F.avg("fare_amount"), 2).alias("fare_promedio"),
        F.round(F.avg("trip_duration_min"), 2).alias("duracion_prom_min"),
    )
    .orderBy("hora")
    .show(24, truncate=False)
)

In [ ]:
# Patrón 2: distribución por día de semana.
# En Spark, F.dayofweek devuelve 1=domingo, 2=lunes, ..., 7=sábado.
# Se agrega una columna de etiqueta para claridad.
dia_labels = {1: "domingo", 2: "lunes", 3: "martes", 4: "miércoles",
              5: "jueves", 6: "viernes", 7: "sábado"}

print("Distribución por día de semana (1=domingo ... 7=sabado en Spark):")
(
    df.groupBy(F.dayofweek("tpep_pickup_datetime").alias("dia_semana"))
    .agg(
        F.count("*").alias("viajes"),
        F.round(F.avg("fare_amount"), 2).alias("fare_promedio"),
    )
    .orderBy("dia_semana")
    .show(7, truncate=False)
)

In [ ]:
# Patrón 3: conteo mensual cruzado con año (2024 y 2025).
# Produce hasta 24 filas (12 meses x 2 años). Los años fuera de rango
# (2002, 2007-2009, 2023, 2026) observados en sección 5.3 aparecerán
# pero con conteos mínimos; se documentan como problema de calidad.
print("Conteo de viajes por año y mes:")
(
    df.groupBy(
        F.year("tpep_pickup_datetime").alias("año"),
        F.month("tpep_pickup_datetime").alias("mes"),
    )
    .agg(F.count("*").alias("viajes"))
    .orderBy("año", "mes")
    .show(30, truncate=False)
)

#### Interpretación de los patrones temporales

**Por hora del día**: el volumen describe una curva diurna clara. El valle nocturno se concentra entre las 3 y 5 AM (mínimo absoluto a las 4 AM con 604,740 viajes); a partir de las 5 AM el volumen crece de manera sostenida y alcanza el máximo absoluto a las 18:00 (6,428,945 viajes). Las horas de la tarde-noche (15:00 a 22:00) concentran el mayor volumen diario. La tarifa promedio máxima ocurre a las 5 AM (USD 24.41), seguida por las 4 AM (USD 21.55): los pocos viajes de madrugada tienden a ser largos y de tarifa alta. La duración promedio del viaje es máxima entre las 14:00 y 16:00 (19.93 a 20.38 minutos), reflejo del tráfico vespertino.

**Por día de semana** (1 = domingo, 7 = sábado en la convención de Spark): el día de mayor volumen es **jueves (5)** con 13,905,489 viajes, seguido por **sábado (7)** con 13,767,655. El día de menor volumen es **lunes (2)** con 11,063,437 viajes; domingo se ubica apenas por encima con 11,726,699. La tarifa promedio más alta se observa en **domingo (USD 19.56)** y **lunes (USD 19.51)**, mientras que **sábado registra la tarifa promedio más baja (USD 17.81)**. Lectura operativa: durante la semana laboral los viajes son rutinarios y predominantemente intra-Manhattan; el sábado, a pesar del alto volumen, baja la tarifa promedio porque los viajes son más cortos y de entretenimiento dentro de la ciudad.

**Por mes y año**: 2024 promedia ~3.4M viajes mensuales (rango 2.96M en enero a 3.83M en octubre); 2025 promedia ~4.0M (rango 3.48M en enero a 4.59M en mayo). El crecimiento interanual ronda 15-20% en cada mes comparable. En ambos años hay caídas en enero y agosto, consistentes con periodos de menor actividad turística y vacaciones en NYC. La tabla incluye además 59 registros con timestamps fuera del rango 2024-2025 (años 2002, 2007, 2008, 2009, 2023 y 2026), que se documentan como problema de calidad temporal a resolver en la etapa siguiente.

### 7.5 Matriz de correlación

Spark MLlib no puede calcular correlaciones directamente sobre columnas sueltas de un DataFrame: sus algoritmos estadísticos y de machine learning esperan que todas las variables de entrada estén empaquetadas en una sola columna de tipo `Vector`. `VectorAssembler` es el transformador de la librería `pyspark.ml.feature` que realiza esta conversión: toma una lista de columnas numéricas (`inputCols`) y genera una nueva columna de tipo `DenseVector` o `SparseVector` (`outputCol`).

El parámetro `handleInvalid="skip"` indica al ensamblador que descarte silenciosamente cualquier fila que contenga un valor nulo en alguna de las columnas de entrada. Esto es necesario aquí porque `cbd_congestion_fee` es nulo para todos los registros de 2024 (aproximadamente 41.2 millones de filas, el 46% del dataset), y otros campos del bloque Flex Fare también presentan nulos. La alternativa `handleInvalid="error"` lanzaría una excepción al encontrar el primer nulo, y `"keep"` sustituiría los nulos por cero, lo que distorsionaría las correlaciones.

Consecuencia importante: la matriz de correlación se calcula sobre el subconjunto de filas sin ningún nulo en las 13 columnas ensambladas. Dado que `cbd_congestion_fee` es nulo en todo 2024, la matriz refleja esencialmente los viajes de 2025 más los viajes no-Flex-Fare de 2024. Este sesgo temporal se debe tener en cuenta al interpretar los coeficientes.

Una vez ensamblado el vector, `Correlation.corr(df_assembled, "features", "pearson")` calcula la matriz de correlación de Pearson en forma distribuida y devuelve un DataFrame de una sola fila con la matriz como `DenseMatrix`. Se convierte a pandas para visualización.

Referencias:
- https://spark.apache.org/docs/latest/ml-statistics.html (sección Correlation)
- https://spark.apache.org/docs/latest/ml-features.html#vectorassembler

In [ ]:
import pandas as pd
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# Columnas a incluir en la matriz de correlación.
# Se incluyen todas las numéricas continuas más trip_duration_min.
# Se excluyen columnas categóricas codificadas (VendorID, RatecodeID,
# payment_type, PULocationID, DOLocationID) y los timestamps.
corr_cols = [
    "passenger_count",
    "trip_distance",
    "trip_duration_min",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
]

# VectorAssembler: empaqueta las columnas en una sola columna "features".
# handleInvalid="skip" descarta filas con cualquier nulo en las columnas
# ensambladas. Esto excluye ~46% de las filas por cbd_congestion_fee (2024)
# y adicionalmente las filas Flex Fare con nulos en otros campos.
assembler = VectorAssembler(
    inputCols=corr_cols,
    outputCol="features",
    handleInvalid="skip",
)

df_assembled = assembler.transform(df).select("features")

# Cálculo de la matriz de correlación de Pearson.
# El resultado es un DataFrame de 1 fila con la DenseMatrix.
corr_matrix_row = Correlation.corr(df_assembled, "features", "pearson").head()
corr_matrix = corr_matrix_row[0].toArray()

# Convertir a pandas para visualización tabular.
corr_pd = pd.DataFrame(corr_matrix, index=corr_cols, columns=corr_cols).round(2)

print(f"Matriz de correlación de Pearson ({len(corr_cols)}x{len(corr_cols)})")
print(f"Nota: calculada sobre filas sin nulos en ninguna de las {len(corr_cols)} columnas.")
print()
print(corr_pd.to_string())

#### Interpretación de la matriz de correlación

La matriz observada **revela un hallazgo metodológico crítico**: las correlaciones que estructuralmente deberían ser altas son cercanas a cero, mientras emergen agrupaciones inesperadas. Esto se explica por la presencia de outliers extremos en `trip_distance` (max 398,608 millas) y en variables monetarias (`fare_amount` max USD 863,372, `total_amount` max USD 863,380): el coeficiente de Pearson es muy sensible a outliers, y un puñado de valores absurdos infla la varianza al punto de enmascarar las relaciones lineales reales.

Correlaciones que deberían ser altas y se ven distorsionadas por outliers:

- `trip_distance` vs `fare_amount`: r = 0.01. Estructuralmente la tarifa se calcula en función de la distancia recorrida; en datos limpios el coeficiente debería superar 0.8.
- `trip_distance` vs `trip_duration_min`: r = 0.03. A mayor distancia mayor duración, salvo por el ruido del tráfico urbano; el r observado refleja la misma distorsión por outliers.
- `fare_amount` vs `tip_amount`: r = 0.07. La propina suele ser un porcentaje de la tarifa para pagos con tarjeta; los outliers la disipan.
- `fare_amount` vs `total_amount`: r = 1.00. Esta sí se sostiene porque `total_amount` es la suma de `fare_amount` más recargos casi fijos, y los outliers afectan ambas variables de forma proporcional.

Correlaciones moderadas-altas que sí emergen incluso sobre datos crudos:

- **Cluster de recargos por zona de congestión**: `improvement_surcharge` vs `congestion_surcharge` r = 0.74; `congestion_surcharge` vs `cbd_congestion_fee` r = 0.60; `improvement_surcharge` vs `cbd_congestion_fee` r = 0.47. Indica que los viajes dentro de la zona de descongestión central de Manhattan acumulan los tres recargos juntos.
- **Cluster de viajes al aeropuerto**: `Airport_fee` vs `tolls_amount` r = 0.45; `Airport_fee` vs `tip_amount` r = 0.41; `tolls_amount` vs `tip_amount` r = 0.45. Los viajes a/desde aeropuertos pasan por peajes, generan tarifas más altas y atraen propinas más generosas.
- **Relación recargos extra**: `extra` vs `Airport_fee` r = 0.30; `extra` vs `improvement_surcharge` r = 0.23; `extra` vs `tolls_amount` r = 0.22. El cargo `extra` (recargos misceláneos) se asocia con los demás recargos del viaje.

Conclusión: esta matriz no es válida como referencia de las relaciones genuinas entre variables; es prueba directa de que la limpieza de outliers en la etapa siguiente es indispensable antes de cualquier modelado supervisado o no supervisado. Una matriz post-limpieza, o el uso del coeficiente de Spearman (basado en rangos y robusto a outliers), debería arrojar valores muy distintos para las correlaciones estructurales esperadas (`trip_distance` vs `fare_amount`, `trip_distance` vs `trip_duration_min`, `fare_amount` vs `tip_amount`).

### Cierre de la sección 7

Los agregados calculados en esta sección quedan disponibles para los pasos siguientes:

- La etapa de visualización utilizará los resultados de `summary()`, las tablas por hora, día y mes, y la matriz de correlaciones para generar gráficos de distribución, mapas de calor y series de tiempo.
- La etapa de calidad de datos partirá de la tabla de nulos, los rangos extremos del resumen estadístico y la presencia de los IDs 264 y 265 en el top de zonas como insumos para definir las reglas de limpieza.

## 8. Problemas de calidad detectados y correcciones propuestas

Esta sección consolida los problemas de calidad observados durante el análisis exploratorio (secciones 5 y 7) y propone las correcciones que se aplicarán en la Etapa 2 (Preprocesamiento). Por requisito de la rúbrica, la Etapa 1 documenta los hallazgos sin ejecutar las correcciones: cada decisión queda sustentada con la evidencia numérica recogida arriba.


| # | Problema observado | Evidencia | Corrección propuesta | Sustento |
|---|---|---|---|---|
| 1 | `trip_distance` con valores físicamente imposibles | Max observado 398,608 millas (sección 5.2); p99 razonable según `approxQuantile` (sección 7.1) | Filtrar `trip_distance` al rango `[0, percentil 99.9]` o a un tope físico de 200 mi (Manhattan-Montauk ida y vuelta); registrar las filas excluidas | Un viaje de 398k millas equivale a 16 vueltas a la Tierra; es ruido de medición o error de captura, no un viaje real |
| 2 | Variables monetarias con outliers de seis cifras | `fare_amount` max USD 863,372; `total_amount` max USD 863,380 (sección 5.2 y 7.1) | Filtrar `fare_amount` y `total_amount` al rango `[0, p99.9]`; reportar el porcentaje de filas removidas | La tarifa más alta razonable en NYC ronda los USD 1,000 (viajes interestatales raros); valores de seis cifras son errores de captura |
| 3 | `passenger_count` con valores fuera del rango documentado | Max observado 9; el manual TLC indica capacidad típica 1-6 (sección 5.2) | Imputar `passenger_count = 1` (moda y mediana del dataset) para los registros con valor 0, 7, 8, 9 o nulo; conservar tal cual los valores 1-6 | La moda y la mediana coinciden en 1 porque la mayoría de viajes en taxi son individuales; imputar a la moda preserva la cardinalidad del dataset sin distorsionar la distribución, a diferencia de la media (~1.4) que no es representable en una variable entera |
| 4 | 59 timestamps fuera del periodo 2024-2025 | Años observados: 2002, 2007, 2008, 2009, 2023, 2026 (sección 7.4 / 7.2) | Filtrar `tpep_pickup_datetime` al rango `[2024-01-01, 2025-12-31]` | El dataset declara cubrir 2024 y 2025; años fuera son errores de reloj del medidor o contaminación cruzada de archivos |
| 5 | 21 registros 2024 con `cbd_congestion_fee` no nulo | Conteo cruzado año vs columna no nula (sección 5.3) | Imputar `cbd_congestion_fee = 0.0` para todo registro con `tpep_pickup_datetime < 2025-01-05` | El cargo CBD entró en vigor el 2025-01-05; cualquier valor previo es un cobro indebido y debe normalizarse a cero, que es el valor coherente con la regulación vigente en esa fecha |
| 6 | Nulos estructurales en `cbd_congestion_fee` (46.4%) | Conteo de nulos sección 7.2; columna ausente en archivos 2024 | Imputar `cbd_congestion_fee = 0.0` para todo registro 2024 (la columna no existía) | La ausencia es estructural, no un faltante aleatorio: en 2024 la columna no formaba parte del esquema TLC |
| 7 | Nulos correlacionados de Flex Fare en columnas tarifarias | Sección 7.2: `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, `Airport_fee` con 17.47% nulos cada una (15,703,126 filas; verificado: son las mismas filas) | Marcar bandera `is_flex_fare = (payment_type == 0)`; no imputar; el tratamiento (incluir o excluir) se decide por tipo de análisis posterior (ver matriz en sección 7.2) | Los registros Flex Fare se reportan sin esos campos por diseño del nuevo modelo tarifario; imputar inventaría telemetría del medidor que nunca existió |
| 8 | Zonas placeholder 264 y 265 en el top-10 | Top zonas (sección 7.3); en `taxi_zone_lookup` corresponden a "Unknown" y "N/A" | Excluir registros con `PULocationID IN (264, 265)` o `DOLocationID IN (264, 265)` para análisis geográficos; conservar para conteos globales | Estos IDs no representan zonas físicas; incluirlos contamina cualquier análisis por zona o ruta |
| 9 | Correlaciones estructurales distorsionadas por outliers | Matriz Pearson sección 7.5: `trip_distance` vs `fare_amount` r = 0.01 (debería ser > 0.8) | Recalcular la matriz tras aplicar las correcciones 1, 2, 3 y 10, o usar el coeficiente de Spearman como contraste robusto | Pearson es muy sensible a outliers extremos; la limpieza de las correcciones 1, 2, 3 y 10 debería restaurar las relaciones lineales esperadas |
| 10 | Viajes con `trip_distance = 0` y `fare_amount > 0` | Cluster vertical visible en el scatterplot de la sección 9.4: alineación densa de puntos en la abscisa cero con tarifas hasta USD 100 | Filtrar `trip_distance > 0` cuando `fare_amount > 0`; alternativamente, imputar `trip_distance` a partir de `fare_amount` usando la tarifa por milla observada (~USD 2.50/mi) | Un viaje con tarifa cobrada implica desplazamiento físico; `distance = 0` señala falla del medidor de odómetro o un viaje cancelado mal facturado, no un trayecto real |

Las correcciones 1, 2, 4 y 10 son destructivas (filtran filas) y se evaluarán de forma conjunta para cuantificar el porcentaje total removido; el objetivo es que la limpieza no descarte más del 1% del dataset. Las correcciones 3, 5, 6 y 7 son imputaciones o banderas y no afectan el tamaño del dataset. La corrección 8 se aplica solo a análisis geográficos y la 9 es una recomendación metodológica para la matriz de correlación.


## 9. Visualizaciones descriptivas

Esta sección presenta cinco gráficas que complementan el análisis estadístico de las secciones anteriores con representaciones visuales. Las gráficas cubren: la distribución univariada de la tarifa base, los patrones temporales de demanda, la comparación de propinas por método de pago, la relación bivariada entre distancia y tarifa, y el nivel de completitud por columna. Cada gráfica se genera a partir de agregaciones calculadas en Spark; solo los datos ya reducidos se transfieren al driver con `.toPandas()`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

### 9.1 Distribución de la tarifa base (`fare_amount`)

El histograma muestra la forma de la distribución de `fare_amount` restringida a valores positivos y al percentil 99. Se usa escala logarítmica en el eje Y para que las barras de baja frecuencia (tarifas altas) sean visibles junto a las de alta frecuencia. Los percentiles p50, p95 y p99 se marcan con líneas verticales para anclar los cuartiles extremos.

Se elige un histograma porque `fare_amount` es una variable continua y la pregunta central es si la distribución es unimodal, tiene cola larga o presenta modos secundarios.

In [ ]:
# Calcular p50, p95, p99 de fare_amount con approxQuantile.
# IMPORTANTE: relativeError=0.001 (no 0.01). Con 0.01 el algoritmo de
# Greenwald-Khanna puede devolver cualquier valor entre el rank 0.98 y 1.00
# para p99, y con outliers extremos (fare_amount max ~USD 863k) terminaba
# regresando el máximo, aplastando el histograma.
fare_quantiles = df.approxQuantile("fare_amount", [0.50, 0.95, 0.99], 0.001)
fare_p50, fare_p95, fare_p99 = fare_quantiles

# Muestra acotada al rango (0, p99] para que el histograma muestre la masa
# de la distribución, no la cola de outliers que ya documentamos en sección 7.
fare_sample = (
    df.select("fare_amount")
    .filter((F.col("fare_amount") > 0) & (F.col("fare_amount") <= fare_p99))
    .sample(fraction=0.002, seed=42)
    .limit(150_000)
    .toPandas()
)

fig, ax = plt.subplots()
ax.hist(fare_sample["fare_amount"], bins=50, color="#2166ac", edgecolor="none")
ax.set_yscale("log")
ax.set_xlabel("Tarifa base (USD)")
ax.set_ylabel("Número de viajes (escala log)")
ax.set_title("Distribución de fare_amount (viajes con tarifa > 0, hasta p99)")

for val, label, ls in [
    (fare_p50, "p50", "--"),
    (fare_p95, "p95", "-."),
    (fare_p99, "p99", ":"),
]:
    ax.axvline(val, color="#d6604d", linestyle=ls, linewidth=1.2, label=f"{label} = ${val:.2f}")

ax.legend(fontsize=9)
ax.text(
    0.99, 0.02,
    f"Fuente: NYC TLC Yellow Taxi 2024-2025  |  muestra: {len(fare_sample):,} viajes",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray"
)
fig.tight_layout()
plt.show()


#### Lectura de la gráfica

La distribución de la tarifa base muestra cuánto se concentran los viajes en rangos bajos frente a la cola derecha. Si la distribución es marcadamente asimétrica, la línea del p50 quedará desplazada hacia la izquierda respecto al centro visual del histograma, lo que indicaría que la mayoría de los viajes son de tarifa reducida y los viajes largos o de tarifa alta son relativamente escasos. La escala logarítmica en Y permite evaluar si esa cola es exponencial o tiene discontinuidades que sugieran tarifas planas (aeropuerto, tarifa mínima).

### 9.2 Demanda por hora del día y día de semana

El heatmap cruza las 24 horas del día (eje Y) con los 7 días de la semana (eje X) y colorea cada celda según el conteo de viajes agregado. Esta representación permite identificar de un vistazo si los picos de demanda coinciden en hora y día, o si los patrones entre días laborables y fin de semana difieren.

Se elige un heatmap porque la variable de interés es una intensidad sobre una cuadrícula bidimensional discreta (hora x día), y la comparación célula a célula es más eficiente visualmente que una serie de líneas superpuestas.

In [ ]:
# Agregar conteo por hora y día de semana en Spark.
# F.dayofweek: 1=domingo, 2=lunes, ..., 7=sábado.
hour_dow = (
    df.groupBy(
        F.hour("tpep_pickup_datetime").alias("hora"),
        F.dayofweek("tpep_pickup_datetime").alias("dia_spark"),
    )
    .count()
    .toPandas()
)

# Pivotar a matriz 24 x 7 (horas en filas, días en columnas).
matriz = hour_dow.pivot(index="hora", columns="dia_spark", values="count")

# Reordenar columnas: Lun(2), Mar(3), Mié(4), Jue(5), Vie(6), Sáb(7), Dom(1)
dow_order = [2, 3, 4, 5, 6, 7, 1]
dow_labels = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
matriz = matriz.reindex(columns=dow_order)
matriz = matriz.fillna(0)

fig, ax = plt.subplots(figsize=(10, 7))
# cmap="coolwarm": azul (frío, baja demanda) a rojo (calor, alta demanda).
im = ax.imshow(matriz.values, aspect="auto", cmap="coolwarm", origin="lower")

# Colorbar con números legibles en miles (sin notación 1e6).
cbar = plt.colorbar(im, ax=ax, label="Número de viajes")
cbar.formatter = mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
cbar.update_ticks()

# Anotar el conteo en cada celda para que el lector tenga el número exacto.
vmax = matriz.values.max()
for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        v = matriz.values[i, j]
        # Texto blanco sobre celdas oscuras (extremos), negro sobre celdas claras (centro).
        color = "white" if (v < 0.25 * vmax or v > 0.75 * vmax) else "black"
        ax.text(j, i, f"{int(v/1000)}k", ha="center", va="center", fontsize=6, color=color)

ax.set_xticks(range(7))
ax.set_xticklabels(dow_labels)
ax.set_yticks(range(24))
ax.set_yticklabels(range(24))
ax.set_xlabel("Día de la semana")
ax.set_ylabel("Hora del día (0 = medianoche)")
ax.set_title("Demanda de viajes por hora y día de semana (2024-2025)")

ax.text(
    0.99, -0.08,
    "Fuente: NYC TLC Yellow Taxi 2024-2025",
    transform=ax.transAxes, ha="right", va="top", fontsize=7, color="gray"
)
fig.tight_layout()
plt.show()


#### Lectura de la gráfica

El heatmap permite identificar si hay franjas horarias con demanda consistentemente alta (por ejemplo, la hora pico de la mañana o la tarde) y si esas franjas se comportan de forma distinta entre días laborables y fin de semana. Las celdas en tonos rojos intensos (calor, paleta coolwarm) señalan los momentos de mayor demanda; las celdas azules (frío) indican horas de baja actividad como la madrugada. El conteo exacto de cada celda se anota en miles para lectura precisa. Un patrón típico en ciudades con transporte nocturno activo muestra un segundo pico en las primeras horas del sábado y domingo que no aparece entre semana.

### 9.3 Propina por método de pago (`tip_amount` según `payment_type`)

Los boxplots muestran la dispersión y los cuartiles de `tip_amount` para cada categoría de `payment_type`. La decodificación del campo es: 1 = Tarjeta de crédito, 2 = Efectivo, 3 = Sin cargo, 4 = Disputa, 5 = Desconocido, 6 = Anulado. Los bigotes se extienden del percentil 5 al percentil 95, y la muestra se acota previamente a `tip_amount` en el rango [0, USD 100] para limitar el efecto de outliers extremos.

Se elige un boxplot comparativo porque la pregunta es de distribución por categoría, y este tipo de gráfica permite ver simultáneamente la mediana, los cuartiles y el rango efectivo de cada grupo.

In [ ]:
# Calcular estadísticas de caja para tip_amount por payment_type.
# approxQuantile opera por columna sobre un subconjunto filtrado; se itera
# sobre cada tipo de pago para mantener cada job pequeño.
#
# IMPORTANTE: relativeError=0.0001 (no 0.01). Con 0.01 el p99 puede caer en
# cualquier rank entre 0.98 y 1.00, y como tip_amount tiene outliers de
# USD 1000+ el resultado se aplastaba contra el max y las cajas quedaban
# invisibles pegadas al cero.

PAYMENT_LABELS = {
    1: "Tarjeta de crédito",
    2: "Efectivo",
    3: "Sin cargo",
    4: "Disputa",
    5: "Desconocido",
    6: "Anulado",
}

REL_ERROR = 0.0001

# Tope físico razonable para propinas en NYC: USD 100 (ya genera holgura
# sobre cualquier propina realista; los valores mayores son outliers).
TIP_HARD_CAP = 100.0

bxpstats = []
for pt, label in PAYMENT_LABELS.items():
    sub = df.filter(
        (F.col("payment_type") == pt) &
        (F.col("tip_amount") >= 0) &
        (F.col("tip_amount") <= TIP_HARD_CAP)
    )
    n = sub.count()
    if n < 10:
        continue
    q = sub.approxQuantile("tip_amount", [0.05, 0.25, 0.50, 0.75, 0.95], REL_ERROR)
    bxpstats.append({
        "label": f"{label}\n(n={n:,})",
        "med": q[2],
        "q1": q[1],
        "q3": q[3],
        "whislo": q[0],
        "whishi": q[4],
        "fliers": [],
    })

fig, ax = plt.subplots(figsize=(10, 6))
ax.bxp(bxpstats, showfliers=False, patch_artist=True,
       boxprops=dict(facecolor="#c6dbef", color="#2166ac"),
       medianprops=dict(color="#d6604d", linewidth=1.8),
       whiskerprops=dict(color="#2166ac"),
       capprops=dict(color="#2166ac"))

# Ajustar ylim al máximo whishi observado más un margen del 15%.
ymax = max(s["whishi"] for s in bxpstats) * 1.15
ax.set_ylim(-0.5, max(ymax, 5))
ax.grid(axis="y", alpha=0.3, linestyle=":")

ax.set_xlabel("Método de pago")
ax.set_ylabel("Propina (USD)")
ax.set_title("Distribución de propinas por método de pago (bigotes p5-p95)")
ax.text(
    0.99, 0.98,
    "Fuente: NYC TLC Yellow Taxi 2024-2025  |  filtro: tip_amount en [0, 100]",
    transform=ax.transAxes, ha="right", va="top", fontsize=7, color="gray"
)
fig.tight_layout()
plt.show()


#### Lectura de la gráfica

El boxplot más informativo es la comparación entre los pagos con tarjeta de crédito y los pagos en efectivo. Si la caja del efectivo colapsa en torno a cero con bigote muy corto, el dato confirma que los taxistas reportan propinas nulas en efectivo (el sistema no las captura). La categoría de tarjeta mostrará una distribución más amplia si los pasajeros dejan propinas variables en el terminal. Las categorías con pocos viajes (Sin cargo, Disputa, Anulado) tendrán cajas estrechas o podrían omitirse si el volumen no es representativo.

### 9.4 Relación entre distancia y tarifa (`trip_distance` vs `fare_amount`)

El diagrama de dispersión muestra cómo se relacionan la distancia recorrida y la tarifa cobrada en una muestra de aproximadamente 100,000 viajes. La opacidad baja (`alpha=0.05`) y los puntos pequeños permiten visualizar la densidad de la nube sin saturar el área.

Se elige un scatterplot porque la pregunta es bivariada continua: queremos ver si la relación es lineal, si hay segmentos con pendiente diferente (posibles tarifas planas como aeropuerto) y cuánta varianza tiene la tarifa para una distancia dada.

In [ ]:
# Fracción calculada para obtener ~100k puntos desde 89.9M filas.
# fraction = 100_000 / 89_892_322 ≈ 0.0011; se pide 0.002 y se limita con limit().
scatter_frac = 0.002

scatter_pd = (
    df.select("trip_distance", "fare_amount")
    .filter(
        (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 50) &
        (F.col("fare_amount") > 0) & (F.col("fare_amount") <= 200)
    )
    .sample(fraction=scatter_frac, seed=42)
    .limit(100_000)
    .toPandas()
)

fig, ax = plt.subplots()
ax.scatter(
    scatter_pd["trip_distance"],
    scatter_pd["fare_amount"],
    s=2, alpha=0.05, color="#2166ac", rasterized=True
)
ax.set_xlabel("Distancia del viaje (millas)")
ax.set_ylabel("Tarifa base (USD)")
ax.set_title(f"Distancia vs. tarifa base  (muestra: {len(scatter_pd):,} viajes)")
ax.text(
    0.99, 0.02,
    "Fuente: NYC TLC Yellow Taxi 2024-2025  |  filtro: distancia 0-50 mi, tarifa 0-200 USD",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray"
)
fig.tight_layout()
plt.show()

#### Lectura de la gráfica

Si la relación fuera puramente proporcional al taxímetro, la nube formaría una banda diagonal estrecha con pendiente constante. En la práctica, es frecuente observar dispersión creciente conforme aumenta la distancia (la varianza de la tarifa crece), así como bandas horizontales en valores fijos que corresponden a tarifas planas (por ejemplo, la tarifa fija al aeropuerto JFK). La densidad de la nube en distancias cortas revela que la mayoría de los viajes son urbanos de menos de diez millas.

### 9.5 Porcentaje de nulos por columna

La gráfica de barras horizontales muestra el porcentaje de registros nulos para cada columna que tiene al menos un nulo. Las columnas sin ningún nulo se omiten para no saturar el eje Y. Las barras están ordenadas de mayor a menor para facilitar la priorización de correcciones.

Se elige una barra horizontal porque la variable categórica (nombre de columna) es el eje de interés, y el texto de los nombres es legible en horizontal sin rotación.

In [ ]:
# counts_pd fue calculado en la sección 7.2 y tiene columna 'pct'.
# Si la variable no está en el entorno, recalcularla igual que en 7.2.
try:
    nulls_plot = counts_pd[counts_pd["pct"] > 0][["pct"]].sort_values("pct", ascending=True)
except NameError:
    # Recalculo de respaldo (mismo patrón que sección 7.2).
    counts_pd_bk = df.summary("count").toPandas().drop(columns=["summary"]).T
    counts_pd_bk.columns = ["non_null"]
    counts_pd_bk["non_null"] = counts_pd_bk["non_null"].astype(int)
    ts_cols_bk = [c for c, t in df.dtypes if t.startswith("timestamp") and c not in counts_pd_bk.index]
    if ts_cols_bk:
        ts_row_bk = df.agg(*[F.count(c).alias(c) for c in ts_cols_bk]).first()
        for c in ts_cols_bk:
            counts_pd_bk.loc[c] = [ts_row_bk[c]]
    counts_pd_bk = counts_pd_bk.reindex(df.columns)
    counts_pd_bk["nulos"] = n_rows - counts_pd_bk["non_null"]
    counts_pd_bk["pct"] = (counts_pd_bk["nulos"] / n_rows * 100).round(2)
    counts_pd_bk.index.name = "columna"
    nulls_plot = counts_pd_bk[counts_pd_bk["pct"] > 0][["pct"]].sort_values("pct", ascending=True)

fig, ax = plt.subplots(figsize=(9, max(4, len(nulls_plot) * 0.45)))
bars = ax.barh(nulls_plot.index, nulls_plot["pct"], color="#2166ac")

for bar, val in zip(bars, nulls_plot["pct"]):
    ax.text(
        val + 0.3, bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}%", va="center", ha="left", fontsize=8
    )

ax.set_xlabel("Porcentaje de nulos (%)")
ax.set_title("Completitud por columna: porcentaje de valores nulos")
ax.set_xlim(0, nulls_plot["pct"].max() * 1.15)
ax.text(
    0.99, 0.01,
    "Fuente: NYC TLC Yellow Taxi 2024-2025",
    transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="gray"
)
fig.tight_layout()
plt.show()

#### Lectura de la gráfica

Las barras más largas identifican las columnas con mayor riesgo de sesgo si se usan en análisis sin imputación previa. Un grupo de columnas con porcentaje similar puede indicar que comparten la misma fuente o que el campo es opcional en ciertos tipos de viaje (por ejemplo, `passenger_count` y `RatecodeID` tienden a estar ausentes en los mismos registros en datasets TLC). Las columnas con porcentaje cercano a 100 merecen evaluación de si conviene excluirlas del modelo; las de porcentaje bajo pueden tratarse con imputación o eliminación de filas sin pérdida significativa.